# Pipeline TFM — Peg Transfer
Ejecuta el pipeline completo (detección → tracking → estados → métricas) sobre videos en Google Drive.

**Antes de correr:** editar las variables en la celda de configuración.

In [ ]:
# ─── 1. Montar Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ─── 2. Configuración ───────────────────────────────────────────────────────
# Repositorio
REPO_URL   = 'https://github.com/jonathanpo7/TFM_laparoscopic.git' 
BRANCH     = 'main'
REPO_LOCAL = '/content/repo'

# Rutas en Drive
VIDEO_DIR   = '/content/drive/MyDrive/TFM/videos'        # carpeta con los .mp4
OUTPUT_DIR  = '/content/drive/MyDrive/TFM/outputs'       # donde guardar los JSON
MODEL_DRIVE = '/content/drive/MyDrive/TFM/xl1280-1.pt'   # modelo en Drive

print('Configuración cargada.')

In [ ]:
# ─── 3. Clonar repo e instalar dependencias ─────────────────────────────────
import subprocess
from pathlib import Path

if not Path(REPO_LOCAL).exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_LOCAL}
else:
    print('Repo ya clonado — haciendo pull...')
    !git -C {REPO_LOCAL} pull

!pip install -q -r {REPO_LOCAL}/requirements.txt
print('Dependencias instaladas.')

In [ ]:
# ─── 4. Copiar modelo al lugar esperado por el pipeline ─────────────────────
import shutil

model_dest = Path(REPO_LOCAL) / 'projects' / 'model' / 'xl1280-1.pt'
model_dest.parent.mkdir(parents=True, exist_ok=True)

if not model_dest.exists():
    shutil.copy(MODEL_DRIVE, model_dest)
    print(f'Modelo copiado: {model_dest}')
else:
    print(f'Modelo ya presente: {model_dest}')

In [ ]:
# ─── 5. Listar videos a procesar ────────────────────────────────────────────
video_dir  = Path(VIDEO_DIR)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

videos = sorted(video_dir.glob('*.mp4'))
print(f'Videos encontrados: {len(videos)}')
for v in videos:
    out = output_dir / f'{v.stem}_metrics.json'
    estado = '[OK]  ' if out.exists() else '[---]'
    print(f'  {estado} {v.name}')

In [ ]:
# ─── 6. Correr pipeline ─────────────────────────────────────────────────────
import sys
sys.path.insert(0, f'{REPO_LOCAL}/projects/main')
sys.path.insert(0, f'{REPO_LOCAL}/projects')

from main import run_pipeline

for video_path in videos:
    out = output_dir / f'{video_path.stem}_metrics.json'
    if out.exists():
        print(f'[SKIP] {video_path.name} — ya procesado')
        continue
    print(f'\n[PROCESANDO] {video_path.name}')
    try:
        run_pipeline(video_path, out)
        print(f'[OK] {out.name}')
    except Exception as e:
        print(f'[ERROR] {video_path.name}: {e}')

print('\n=== Listo ===')

In [ ]:
# ─── 7. Resumen de salidas ──────────────────────────────────────────────────
import json

jsons = sorted(output_dir.glob('*_metrics.json'))
print(f'JSONs generados: {len(jsons)}\n')
for j in jsons:
    with open(j) as f:
        m = json.load(f)
    s = m.get('summary', {})
    print(f"{j.stem}")
    print(f"  Rings: {s.get('rings_completed')}/{s.get('rings_total')}  "
          f"Tiempo: {s.get('total_exercise_time_s')}s")